# Serve Qwen2.5-3B-Instruct for Stock Multi-Agent

Chọn **Runtime → Change runtime type → T4 GPU**, sau đó **Run all**. Cell cuối sẽ in `COLAB_QWEN_BASE_URL` và `COLAB_QWEN_API_TOKEN` để dán vào `.env` trên máy local. Giữ tab Colab và runtime hoạt động trong lúc dùng app.

In [ ]:
!python -m pip install -q -U 'transformers>=4.48,<5' 'accelerate>=1.2,<2' 'fastapi>=0.115,<1' 'uvicorn>=0.34,<1' 'jsonschema>=4.23,<5'


In [ ]:
from __future__ import annotations

import json
import re
import secrets
import threading
from typing import Any

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
if not torch.cuda.is_available():
    raise RuntimeError('Hãy bật GPU T4 trong Runtime settings trước khi chạy.')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map='auto', low_cpu_mem_usage=True,
)
model.eval()
print('Loaded', MODEL_ID, '| GPU:', torch.cuda.get_device_name(0))


In [ ]:
from fastapi import FastAPI, Header, HTTPException
from jsonschema import validate as validate_json
from pydantic import BaseModel, Field

API_TOKEN = secrets.token_urlsafe(32)
INFERENCE_LOCK = threading.Lock()
app = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)

class GenerateRequest(BaseModel):
    model: str | None = None
    system: str
    payload: dict[str, Any]
    json_schema: dict[str, Any] | None = None
    max_new_tokens: int = Field(default=512, ge=64, le=1024)

def authorize(value: str | None) -> None:
    if value != f'Bearer {API_TOKEN}':
        raise HTTPException(status_code=401, detail='Unauthorized')

def generate_text(system: str, user_text: str, max_new_tokens: int) -> str:
    messages = [{'role': 'system', 'content': system}, {'role': 'user', 'content': user_text}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with INFERENCE_LOCK, torch.inference_mode():
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0, inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

def extract_json(text: str) -> dict[str, Any]:
    cleaned = re.sub(r'^```(?:json)?\s*|\s*```$', '', text.strip(), flags=re.I | re.S)
    start, end = cleaned.find('{'), cleaned.rfind('}')
    if start < 0 or end <= start:
        raise ValueError('No JSON object found')
    value = json.loads(cleaned[start:end + 1])
    if not isinstance(value, dict):
        raise ValueError('JSON output must be an object')
    return value

@app.get('/health')
def health():
    return {'status': 'ok', 'model': MODEL_ID}

@app.post('/generate-structured')
def generate_structured(request: GenerateRequest, authorization: str | None = Header(default=None)):
    authorize(authorization)
    if request.model and request.model != MODEL_ID:
        raise HTTPException(status_code=400, detail='Unsupported model')
    schema = request.json_schema or {'type': 'object'}
    user_text = (
        'Input facts:\n' + json.dumps(request.payload, ensure_ascii=False)
        + '\nRequired JSON Schema:\n' + json.dumps(schema, ensure_ascii=False)
        + '\nReturn one JSON object only. Do not use markdown fences.'
    )
    last_error = 'invalid structured output'
    for attempt in range(2):
        suffix = '' if attempt == 0 else '\nPrevious output was invalid. Repair it and obey the schema exactly.'
        try:
            parsed = extract_json(generate_text(request.system, user_text + suffix, request.max_new_tokens))
            validate_json(instance=parsed, schema=schema)
            return {'output': parsed, 'model': MODEL_ID}
        except Exception as exc:
            last_error = type(exc).__name__
    raise HTTPException(status_code=422, detail=f'Model output failed schema validation: {last_error}')

@app.post('/generate')
def generate(request: GenerateRequest, authorization: str | None = Header(default=None)):
    authorize(authorization)
    text = generate_text(request.system, json.dumps(request.payload, ensure_ascii=False), request.max_new_tokens)
    return {'output': text, 'model': MODEL_ID}


In [ ]:
import subprocess
import time
import urllib.request
from pathlib import Path
import uvicorn

server_thread = threading.Thread(
    target=lambda: uvicorn.run(app, host='127.0.0.1', port=8000, log_level='warning'),
    daemon=True,
)
server_thread.start()
for _ in range(30):
    try:
        if json.loads(urllib.request.urlopen('http://127.0.0.1:8000/health').read())['status'] == 'ok':
            break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('FastAPI server did not start')

!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared
tunnel_log_path = Path('/content/cloudflared.log')
tunnel_log = tunnel_log_path.open('w', encoding='utf-8')
tunnel = subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=tunnel_log, stderr=subprocess.STDOUT, text=True,
)
public_url = None
deadline = time.time() + 45
while time.time() < deadline and public_url is None:
    tunnel_log.flush()
    log_text = tunnel_log_path.read_text(encoding='utf-8', errors='replace')
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log_text)
    if match:
        public_url = match.group(0)
    if tunnel.poll() is not None:
        break
    time.sleep(1)
if not public_url:
    raise RuntimeError('Cloudflare tunnel did not provide a public URL')
print('\nCopy these two lines into your local .env:\n')
print('COLAB_QWEN_BASE_URL=' + public_url)
print('COLAB_QWEN_API_TOKEN=' + API_TOKEN)
print('\nKeep this Colab runtime running. The URL/token changes after every restart.')
